# Robustness Stress Test of the Minimal Event-Based Monte Carlo Model

## Problem statement

This notebook continues the existing reduced 2D event-based model for lateral interaction between two elongated floating bodies.

The question in this iteration is narrower than physical truth:

> Does the apparent attraction seen in the first prototype survive broader testing, or is it mainly an artifact of specific modeling assumptions?

The notebook keeps four things separate:

- observations from the simulation
- assumptions built into the toy model
- sensitivity of the results to those assumptions
- physical inference that remains unjustified at this stage

The implementation logic lives in `src/float_sim/event_model.py`. The notebook is the experiment driver and interpretation layer.


## Brief recap of the current minimal model

The existing prototype already established two useful control facts:

- with blocking disabled, the mean gap-closing force is negative across the initial gap sweep, so attraction is **not** baked into the model
- with blocking enabled, the same model can show positive gap-closing force at small gaps

That is not proof of the original physical hypothesis. It only shows that the current **binary line-of-sight shielding rule** can create attraction in this reduced setting.

The model remains intentionally simple:

- two parallel rectangles in a 2D top view
- stochastic wave events sampled in a bounded source box
- explicit attenuation with distance
- impulses projected onto inner and outer side normals
- optional blocking when a straight segment from source to side sample intersects the other body

The goal here is therefore **robustness analysis, not realism inflation**.


In [ ]:
from dataclasses import replace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.append(str(SRC_ROOT))

from float_sim.event_model import (
    ModelParameters,
    SourceField,
    event_count_from_source_density,
    plot_ensemble_metric,
    plot_geometry,
    plot_inner_outer_summary,
    plot_side_metrics,
    run_ensemble,
    run_gap_ensemble_sweep,
    simulate_batch,
    summarize_ensemble,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120
np.set_printoptions(precision=3, suppress=True)


## Why this iteration focuses on robustness rather than realism

A more realistic wave solver would make it harder to tell why attraction appears or disappears. The present model is still useful because it can separate several competing explanations:

- fewer inner-side hits versus fewer inner-side impulse totals
- direct effects of the blocking rule versus effects that survive without blocking
- finite source-box geometry versus local body geometry
- source anisotropy versus symmetric forcing

If attraction only appears in a narrow region, or only under one assumption such as binary blocking, the right conclusion is fragility, not confirmation.


In [ ]:
BASE_SEED = 240
ENSEMBLE_REPEATS = 8
SLICE_REPEATS = 6
BASE_SOURCE_DENSITY = 8.0
GAPS = np.array([0.1, 0.3, 0.6, 1.0, 1.4, 1.8])
REFERENCE_GAPS = (0.3, 1.0, 1.8)

UNIFORM_FIELD = SourceField()
BIASED_FIELD = SourceField(model="vertical_gradient", vertical_bias=0.7)

BASE_PARAMS = ModelParameters(
    body_length=3.0,
    body_width=0.4,
    domain_half_length=6.0,
    domain_half_width=4.0,
    attenuation_length=2.5,
    attenuation_power=0.0,
    mean_wave_amplitude=1.0,
    side_samples=17,
    mobility=0.002,
)


def events_for(params: ModelParameters, density: float = BASE_SOURCE_DENSITY) -> int:
    return event_count_from_source_density(density, params)


def gap_sweep(
    *,
    params: ModelParameters = BASE_PARAMS,
    source_density: float = BASE_SOURCE_DENSITY,
    blocking_enabled: bool,
    source_field: SourceField,
    repeats: int = ENSEMBLE_REPEATS,
    seed: int = BASE_SEED,
):
    return run_gap_ensemble_sweep(
        gaps=GAPS,
        params=params,
        n_events=events_for(params, source_density),
        repeats=repeats,
        seed=seed,
        blocking_enabled=blocking_enabled,
        source_field=source_field,
    )


def ensemble_at_gap(
    gap: float,
    *,
    params: ModelParameters = BASE_PARAMS,
    source_density: float = BASE_SOURCE_DENSITY,
    blocking_enabled: bool,
    source_field: SourceField,
    repeats: int = ENSEMBLE_REPEATS,
    seed: int = BASE_SEED,
):
    return run_ensemble(
        gap=gap,
        params=params,
        n_events=events_for(params, source_density),
        repeats=repeats,
        seed=seed,
        blocking_enabled=blocking_enabled,
        source_field=source_field,
    )


def compact_summary_table(summaries):
    rows = []
    for summary in summaries:
        rows.append(
            {
                "gap": round(summary.gap, 2),
                "force": round(summary.mean_gap_closing_force, 2),
                "sem": round(summary.sem_gap_closing_force, 2),
                "system_net": round(summary.mean_system_net_force_y, 2),
                "inner_hits": round(summary.mean_inner_hits, 1),
                "outer_hits": round(summary.mean_outer_hits, 1),
                "inner_impulse": round(summary.mean_inner_impulse, 2),
                "outer_impulse": round(summary.mean_outer_impulse, 2),
            }
        )
    return rows


def parameter_slice(
    values,
    builder,
    *,
    gaps=REFERENCE_GAPS,
    source_density: float = BASE_SOURCE_DENSITY,
    blocking_enabled: bool = True,
    source_field: SourceField = UNIFORM_FIELD,
    repeats: int = SLICE_REPEATS,
    seed: int = BASE_SEED,
):
    rows = []
    for value_index, value in enumerate(values):
        params = builder(value)
        n_events = events_for(params, source_density)
        for gap_index, gap in enumerate(gaps):
            summary = summarize_ensemble(
                run_ensemble(
                    gap=gap,
                    params=params,
                    n_events=n_events,
                    repeats=repeats,
                    seed=seed + 100 * value_index + 11 * gap_index,
                    blocking_enabled=blocking_enabled,
                    source_field=source_field,
                )
            )
            rows.append((float(value), float(gap), summary))
    return rows


def density_slice(
    densities,
    *,
    params: ModelParameters = BASE_PARAMS,
    gaps=REFERENCE_GAPS,
    blocking_enabled: bool = True,
    source_field: SourceField = UNIFORM_FIELD,
    repeats: int = SLICE_REPEATS,
    seed: int = BASE_SEED,
):
    rows = []
    for density_index, density in enumerate(densities):
        n_events = events_for(params, density)
        for gap_index, gap in enumerate(gaps):
            summary = summarize_ensemble(
                run_ensemble(
                    gap=gap,
                    params=params,
                    n_events=n_events,
                    repeats=repeats,
                    seed=seed + 100 * density_index + 13 * gap_index,
                    blocking_enabled=blocking_enabled,
                    source_field=source_field,
                )
            )
            rows.append((float(density), float(gap), summary))
    return rows


def plot_parameter_slice(ax, rows, *, xlabel: str, title: str):
    colors = {0.3: "#e76f51", 1.0: "#457b9d", 1.8: "#1d3557"}
    for gap in REFERENCE_GAPS:
        subset = [(value, summary) for value, row_gap, summary in rows if np.isclose(row_gap, gap)]
        xs = np.array([value for value, _ in subset], dtype=float)
        ys = np.array([summary.mean_gap_closing_force for _, summary in subset], dtype=float)
        es = np.array([summary.sem_gap_closing_force for _, summary in subset], dtype=float)
        ax.plot(xs, ys, marker="o", color=colors[gap], label=f"gap = {gap:.1f}")
        ax.fill_between(xs, ys - es, ys + es, color=colors[gap], alpha=0.15)
    ax.axhline(0.0, color="0.4", linestyle="--", linewidth=1.0)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Mean gap-closing force")
    ax.set_title(title)
    ax.legend(loc="best")


def print_named_table(name, summaries):
    print(name)
    for row in compact_summary_table(summaries):
        print(row)
    print()


## Parameters varied in this round

The stress test uses deterministic seed ranges and repeats each configuration over multiple independent Monte Carlo draws.

Parameter sweeps included here:

- edge-to-edge gap
- attenuation length
- body length
- body width
- source box size
- source density
- blocking on and off
- two source-field models: uniform and vertically biased

The main observables are:

- mean gap-closing force
- mean system net lateral force
- inner-side versus outer-side hit counts
- inner-side versus outer-side cumulative impulse
- distribution of run-to-run gap-closing force and impulse imbalance for selected cases


In [ ]:
reference_batch = simulate_batch(
    gap=0.6,
    params=BASE_PARAMS,
    rng=np.random.default_rng(BASE_SEED),
    n_events=events_for(BASE_PARAMS),
    blocking_enabled=True,
    source_field=UNIFORM_FIELD,
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
plot_geometry(axes[0], reference_batch, max_events=500)
plot_side_metrics(axes[1], reference_batch, metric="hits")
plot_side_metrics(axes[2], reference_batch, metric="impulse")
fig.suptitle("Representative blocked batch from the existing minimal model", fontsize=14)
fig.tight_layout()
plt.show()

print(
    {
        "source_density": BASE_SOURCE_DENSITY,
        "events_per_run": events_for(BASE_PARAMS),
        "mean_gap_closing_force": round(reference_batch.mean_gap_closing_force, 2),
        "mean_inner_hits": round(0.5 * (reference_batch.upper.inner.hits + reference_batch.lower.inner.hits), 1),
        "mean_outer_hits": round(0.5 * (reference_batch.upper.outer.hits + reference_batch.lower.outer.hits), 1),
        "mean_inner_impulse": round(
            0.5 * (
                reference_batch.upper.inner.cumulative_impulse
                + reference_batch.lower.inner.cumulative_impulse
            ),
            2,
        ),
        "mean_outer_impulse": round(
            0.5 * (
                reference_batch.upper.outer.cumulative_impulse
                + reference_batch.lower.outer.cumulative_impulse
            ),
            2,
        ),
    }
)


## Results: force vs gap under multiple settings

The first question is whether the earlier pattern survives ensemble averaging and whether it depends mainly on blocking, source anisotropy, or both.


In [ ]:
uniform_free = gap_sweep(blocking_enabled=False, source_field=UNIFORM_FIELD)
uniform_blocked = gap_sweep(blocking_enabled=True, source_field=UNIFORM_FIELD)
biased_free = gap_sweep(blocking_enabled=False, source_field=BIASED_FIELD)
biased_blocked = gap_sweep(blocking_enabled=True, source_field=BIASED_FIELD)

fig, axes = plt.subplots(2, 2, figsize=(13.5, 10), sharex=True)

plot_ensemble_metric(
    axes[0, 0],
    uniform_free,
    metric="gap_closing_force",
    label="uniform, no blocking",
    color="#e76f51",
)
plot_ensemble_metric(
    axes[0, 0],
    uniform_blocked,
    metric="gap_closing_force",
    label="uniform, blocking",
    color="#1d3557",
)
axes[0, 0].set_title("Uniform source field")

plot_ensemble_metric(
    axes[0, 1],
    biased_free,
    metric="gap_closing_force",
    label="biased, no blocking",
    color="#f4a261",
)
plot_ensemble_metric(
    axes[0, 1],
    biased_blocked,
    metric="gap_closing_force",
    label="biased, blocking",
    color="#2a9d8f",
)
axes[0, 1].set_title("Biased source field")

plot_ensemble_metric(
    axes[1, 0],
    uniform_free,
    metric="system_net_force",
    label="uniform, no blocking",
    color="#e76f51",
)
plot_ensemble_metric(
    axes[1, 0],
    uniform_blocked,
    metric="system_net_force",
    label="uniform, blocking",
    color="#1d3557",
)
axes[1, 0].set_title("Uniform forcing: system net lateral force")

plot_ensemble_metric(
    axes[1, 1],
    biased_free,
    metric="system_net_force",
    label="biased, no blocking",
    color="#f4a261",
)
plot_ensemble_metric(
    axes[1, 1],
    biased_blocked,
    metric="system_net_force",
    label="biased, blocking",
    color="#2a9d8f",
)
axes[1, 1].set_title("Biased forcing: system net lateral force")

fig.tight_layout()
plt.show()

print_named_table("Uniform, no blocking", uniform_free)
print_named_table("Uniform, blocking enabled", uniform_blocked)
print_named_table("Biased, no blocking", biased_free)
print_named_table("Biased, blocking enabled", biased_blocked)


## Inner versus outer diagnostics

Attraction in this model can come from fewer inner-side hits, lower inner-side cumulative impulse, or both. The next plots show which side is favored under the main control pair: symmetric uniform forcing with blocking off versus on.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13.5, 9.5), sharex=True)

plot_inner_outer_summary(axes[0, 0], uniform_free, metric="hits")
axes[0, 0].set_title("Uniform, no blocking: inner vs outer hits")

plot_inner_outer_summary(axes[0, 1], uniform_blocked, metric="hits")
axes[0, 1].set_title("Uniform, blocking enabled: inner vs outer hits")

plot_inner_outer_summary(axes[1, 0], uniform_free, metric="impulse")
axes[1, 0].set_title("Uniform, no blocking: inner vs outer impulse")

plot_inner_outer_summary(axes[1, 1], uniform_blocked, metric="impulse")
axes[1, 1].set_title("Uniform, blocking enabled: inner vs outer impulse")

fig.tight_layout()
plt.show()


In [ ]:
distribution_cases = {
    "uniform / no block / gap 0.3": ensemble_at_gap(0.3, blocking_enabled=False, source_field=UNIFORM_FIELD),
    "uniform / block / gap 0.3": ensemble_at_gap(0.3, blocking_enabled=True, source_field=UNIFORM_FIELD),
    "uniform / block / gap 1.8": ensemble_at_gap(1.8, blocking_enabled=True, source_field=UNIFORM_FIELD),
    "biased / block / gap 0.3": ensemble_at_gap(0.3, blocking_enabled=True, source_field=BIASED_FIELD),
}

labels = list(distribution_cases)
force_data = [[record.mean_gap_closing_force for record in distribution_cases[label]] for label in labels]
imbalance_data = [[record.impulse_imbalance for record in distribution_cases[label]] for label in labels]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
axes[0].boxplot(force_data, tick_labels=labels, showmeans=True)
axes[0].axhline(0.0, color="0.4", linestyle="--", linewidth=1.0)
axes[0].set_ylabel("Per-run mean gap-closing force")
axes[0].set_title("Run-to-run force distributions")
axes[0].tick_params(axis="x", rotation=18)

axes[1].boxplot(imbalance_data, tick_labels=labels, showmeans=True)
axes[1].axhline(0.0, color="0.4", linestyle="--", linewidth=1.0)
axes[1].set_ylabel("Outer minus inner impulse")
axes[1].set_title("Run-to-run impulse-imbalance distributions")
axes[1].tick_params(axis="x", rotation=18)

fig.tight_layout()
plt.show()


## Source-field sensitivity and finite-domain sensitivity

Two separate questions matter here:

1. Does an anisotropic source field create attraction by itself?
2. Does the finite source box change the sign or magnitude of the interaction?

The biased source field is intentionally simple: a vertical density gradient that favors positive `y`. It is useful because it breaks symmetry without changing the attenuation law or blocking rule.


In [ ]:
domain_scales = [0.75, 1.0, 1.5, 2.0]

blocked_domain_slice = parameter_slice(
    domain_scales,
    lambda scale: replace(
        BASE_PARAMS,
        domain_half_length=BASE_PARAMS.domain_half_length * scale,
        domain_half_width=BASE_PARAMS.domain_half_width * scale,
    ),
    blocking_enabled=True,
    source_field=UNIFORM_FIELD,
)
free_domain_slice = parameter_slice(
    domain_scales,
    lambda scale: replace(
        BASE_PARAMS,
        domain_half_length=BASE_PARAMS.domain_half_length * scale,
        domain_half_width=BASE_PARAMS.domain_half_width * scale,
    ),
    blocking_enabled=False,
    source_field=UNIFORM_FIELD,
)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))
plot_parameter_slice(
    axes[0],
    blocked_domain_slice,
    xlabel="Domain scale factor",
    title="Blocking enabled: source-box sensitivity",
)
plot_parameter_slice(
    axes[1],
    free_domain_slice,
    xlabel="Domain scale factor",
    title="No blocking control: source-box sensitivity",
)
fig.tight_layout()
plt.show()


## Geometry, attenuation, and source-density sensitivity

The next slices vary one assumption at a time around the baseline model. Each panel tracks three representative gaps:

- `0.3`: small-gap regime where attraction looked strongest
- `1.0`: intermediate gap
- `1.8`: large-gap regime near the sign change


In [ ]:
attenuation_slice = parameter_slice(
    [1.2, 2.5, 4.5],
    lambda value: replace(BASE_PARAMS, attenuation_length=value),
)
length_slice = parameter_slice(
    [2.0, 3.0, 4.5],
    lambda value: replace(BASE_PARAMS, body_length=value),
)
width_slice = parameter_slice(
    [0.2, 0.4, 0.8],
    lambda value: replace(BASE_PARAMS, body_width=value),
)
density_rows = density_slice([4.0, 8.0, 12.0])

fig, axes = plt.subplots(2, 2, figsize=(13.5, 9.8))
plot_parameter_slice(axes[0, 0], attenuation_slice, xlabel="Attenuation length", title="Attenuation sweep")
plot_parameter_slice(axes[0, 1], length_slice, xlabel="Body length", title="Body-length sweep")
plot_parameter_slice(axes[1, 0], width_slice, xlabel="Body width", title="Body-width sweep")
plot_parameter_slice(axes[1, 1], density_rows, xlabel="Source density", title="Source-density sweep")
fig.tight_layout()
plt.show()


## Control cases

The controls below are meant to test expectations that should hold if the code is not simply manufacturing attraction:

- without blocking and under symmetric forcing, systematic attraction should not be robust
- enlarging the symmetric source box should reduce some box-edge artifacts
- under symmetric forcing, the **system net lateral force** should remain small relative to the interaction force because the two-body setup is symmetric on average


In [ ]:
large_domain_params = replace(BASE_PARAMS, domain_half_length=12.0, domain_half_width=8.0)

control_summaries = {
    "uniform, no blocking, base box, gap 0.3": summarize_ensemble(
        ensemble_at_gap(0.3, blocking_enabled=False, source_field=UNIFORM_FIELD)
    ),
    "uniform, no blocking, large box, gap 0.3": summarize_ensemble(
        ensemble_at_gap(
            0.3,
            params=large_domain_params,
            blocking_enabled=False,
            source_field=UNIFORM_FIELD,
            repeats=SLICE_REPEATS,
        )
    ),
    "uniform, no blocking, large box, gap 1.8": summarize_ensemble(
        ensemble_at_gap(
            1.8,
            params=large_domain_params,
            blocking_enabled=False,
            source_field=UNIFORM_FIELD,
            repeats=SLICE_REPEATS,
        )
    ),
    "uniform, blocking, base box, gap 0.3": summarize_ensemble(
        ensemble_at_gap(0.3, blocking_enabled=True, source_field=UNIFORM_FIELD)
    ),
}

for label, summary in control_summaries.items():
    print(label)
    print(
        {
            "force": round(summary.mean_gap_closing_force, 2),
            "sem": round(summary.sem_gap_closing_force, 2),
            "system_net": round(summary.mean_system_net_force_y, 2),
            "inner_impulse": round(summary.mean_inner_impulse, 2),
            "outer_impulse": round(summary.mean_outer_impulse, 2),
        }
    )
    print()


## Robustness Assessment

The current evidence does **not** support the original physical hypothesis in a strong form.

What the stress tests do support is a weaker and more conditional statement:

- attraction appears consistently at small gaps in this reduced model **when binary blocking is enabled**
- the same attraction disappears under the baseline no-blocking control, so the blocking rule is not a detail; it is a decisive assumption
- attraction at intermediate gaps survives several geometry, attenuation, density, and source-model variations, but its magnitude changes substantially
- attraction at larger gaps is fragile: the sign can move toward zero or turn negative depending on attenuation length, body width, body length, and source density
- finite source-domain geometry matters strongly: enlarging the source box can change the sign at large gaps and can even remove small-gap repulsion in the no-blocking control
- the vertically biased source field mainly introduces whole-system drift; it does **not** create attraction by itself when blocking is off

Taken together, the most decisive assumptions in the current model are:

- the binary blocking logic
- the size of the finite source domain
- secondarily, attenuation length and body geometry near the large-gap sign-change region

The most defensible interpretation at this stage is therefore:

- the original hypothesis is **not** established by this notebook
- a weaker version is supported only in the sense that a simple shielding rule can generate attraction over a limited gap range in a reduced model
- the evidence remains too assumption-sensitive to claim that real floating bodies should attract for the same reason


## Limits of the current model

- The blocking rule is binary and geometric. It does not represent graded transmission, diffraction, or reflected waves.
- The source field is still an imposed stochastic process rather than a self-consistent wave field generated by the bodies.
- The source domain is rectangular and finite, which demonstrably affects the results.
- The net interaction is inferred from side-impulse bookkeeping, not from a calibrated hydrodynamic force law.
- Only parallel rectangular bodies are studied here.

These limits matter because the robustness study shows that some conclusions change when assumptions change.


## Recommended next step

The next sensible step is **not** higher-complexity fluid simulation. It is to replace the current binary blocking rule with one additional reduced alternative, such as a graded occlusion or partial-transmission rule, and then rerun the same ensemble notebook.

That would directly test whether the present attraction is specific to on/off masking or whether it survives a nearby but still interpretable shielding model.
